# Exercice 5 — Pipeline de prétraitement

In [108]:
import numpy as np

import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.compose import make_column_transformer 

from sklearn.preprocessing import OneHotEncoder ,  StandardScaler

from sklearn.impute import  SimpleImputer 

from sklearn.pipeline import make_pipeline 



### Importation du dataset

In [109]:
df = pd.read_csv("dataset/ex1_dataset_non_traité.csv")

df = df.reset_index(drop=True)

df

,age,salaire_annuel,taille_cm,niveau_education,ville,genre,statut_marital
0,38.0,35519.0,171.0,Lycée,Nantes,NaN,NaN
1,22.0,42231.0,174.0,Lycée,Nantes,Femme,Célibataire
2,44.0,61821.0,177.6,Lycée,Paris,Femme,Marié
3,46.0,NaN,NaN,Bac+5,Nice,Homme,Célibataire
4,18.0,NaN,NaN,Lycée,NaN,Femme,Célibataire
...,...,...,...,...,...,...,...
195,24.0,39530.0,194.1,Bac+2,Toulouse,Femme,Célibataire
196,42.0,74686.0,188.2,Bac+2,Marseille,NaN,Célibataire
197,30.0,83456.0,168.3,Bac+5,Nice,Homme,Marié
198,34.0,81396.0,168.7,Bac+5,Nantes,Femme,Célibataire


### Separation du train et du test 

In [110]:
x_train , x_test  = train_test_split(df,test_size=0.3)


### Construction un ColumnTransformer combinant deux sous-pipelines 

In [111]:
num_col = ["age","salaire_annuel","taille_cm"]

cat_col = ["niveau_education","ville","genre","statut_marital"]


num_pipeline = make_pipeline(
    
    ( SimpleImputer(strategy="median") ),

    ( StandardScaler() ),

)


cat_pipeline = make_pipeline(

    ( SimpleImputer(strategy="most_frequent") ),
    
    ( OneHotEncoder(handle_unknown="ignore")  ),

)


### Encapsulation de ce ColumnTransformer dans un Pipeline complet

In [112]:
prepro = make_column_transformer(

    (num_pipeline , num_col),

    (cat_pipeline , cat_col),

    remainder="drop"

)


### Encapsulation de ce ColumnTransformer dans un Pipeline complet.

In [113]:
pipeline = make_pipeline(
    ( prepro)
)

### Application de fit_transform sur les données d'entraînement et seulement transform sur les données de test.

In [114]:
x_train =  pipeline.fit_transform(x_train)

x_train

array([[ 1.89288476,  1.64976053, -0.34318227, ...,  0.        ,
         1.        ,  0.        ],
       [-0.46096157, -1.61263371, -0.57444831, ...,  1.        ,
         0.        ,  0.        ],
       [ 0.71596159, -1.43597603,  0.39686903, ...,  1.        ,
         0.        ,  0.        ],
       ...,
       [-1.04942315, -0.21038415, -0.26223916, ...,  0.        ,
         0.        ,  0.        ],
       [-0.75519236, -1.17182374, -1.77703168, ...,  0.        ,
         0.        ,  0.        ],
       [-1.53980781, -0.86996401,  0.96347081, ...,  0.        ,
         0.        ,  0.        ]], shape=(140, 21))

In [115]:
x_test =  pipeline.fit_transform(x_test)

x_test

array([[-0.06851824, -1.74499588,  1.05139836, ...,  0.        ,
         0.        ,  0.        ],
       [ 0.14230711,  0.45099976,  0.08496804, ...,  0.        ,
         1.        ,  0.        ],
       [-0.06851824, -0.67638962,  0.31198187, ...,  0.        ,
         1.        ,  0.        ],
       ...,
       [-0.06851824, -0.11406812, -0.53121238, ...,  0.        ,
         0.        ,  0.        ],
       [ 1.30184649, -0.304255  ,  1.29787053, ...,  0.        ,
         1.        ,  0.        ],
       [-0.06851824,  0.2126984 , -0.09015692, ...,  1.        ,
         0.        ,  0.        ]], shape=(60, 20))

### Vérification des statistiques apprises sur TRAIN

In [116]:
scaler = pipeline.named_steps["columntransformer"] \
    .named_transformers_["pipeline-1"] \
    .named_steps["standardscaler"]

print("Moyennes apprises sur TRAIN :", scaler.mean_)
print("Écarts-types appris sur TRAIN :", scaler.scale_)

Moyennes apprises sur TRAIN : [3.36500000e+01 6.44647667e+04 1.72695000e+02]
Écarts-types appris sur TRAIN : [9.48652553e+00 1.89340084e+04 7.70878125e+00]
